In [1]:
!pip install -q pandas numpy scikit-learn
import time
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
from google.colab import files
import io
print("=== DATASET PRINCIPAL ===")
df = pd.read_csv('recommendation_dataset.csv', sep=';', index_col=0)
print(f"Dimensiones: {df.shape} (usuarios x restaurantes)")

# Conversión del 99 a Nan para no afectar los cálculos
df_clean = df.replace(99, np.nan)

# Estadísticas clave
global_mean = np.nanmean(df_clean.values)
user_means = df_clean.mean(axis=1)
item_means = df_clean.mean(axis=0)

print(f"\n=== ESTADÍSTICAS BÁSICAS ===")
print(f"Media global: {global_mean:.4f}")
print(f"Densidad de datos: {(1 - df_clean.isna().mean().mean()) * 100:.2f}%")
print(f"Valoraciones por usuario (media): {df_clean.count(axis=1).mean():.1f}")
print(f"Usuarios sin valoraciones: {user_means.isna().sum()}")
print(f"Restaurantes sin valoraciones: {item_means.isna().sum()}")

# Mostrar primeras medias de usuario y restaurante
print(f"\nPrimeras medias de usuario: {user_means.head(3).tolist()}")
print(f"Primeras medias de restaurante: {item_means.head(3).tolist()}")

=== DATASET PRINCIPAL ===
Dimensiones: (73421, 100) (usuarios x restaurantes)

=== ESTADÍSTICAS BÁSICAS ===
Media global: 0.7419
Densidad de datos: 55.79%
Valoraciones por usuario (media): 55.8
Usuarios sin valoraciones: 0
Restaurantes sin valoraciones: 0

Primeras medias de usuario: [-3.4318918918918917, 2.6666326530612245, 7.095624999999999]
Primeras medias de restaurante: [0.903260039748888, 0.16513797446613615, 0.19747020280811234]


In [6]:
print("=== MÉTODO 8: KNN (k=14) CON DOBLE NORMALIZACIÓN (Similitud del coseno) ===")

start_time = time.time()

print("Calculando sesgos globales...")
R_df = df_clean.copy()
global_mean = R_df.stack().mean()
user_means = R_df.mean(axis=1)
item_means = R_df.mean(axis=0)
u_bias_dict = (user_means - global_mean).to_dict()
i_bias_dict = (item_means - global_mean).to_dict()

print("Aplicando doble normalización a la matriz...")
R_norm_user = R_df.sub(user_means, axis=0)
R_double_norm = R_norm_user.sub(item_means - global_mean, axis=1)

print("Calculando similitud coseno sobre la matriz normalizada...")
R_items = R_double_norm.T.fillna(0.0)
cosine_sim_matrix = cosine_similarity(R_items.values)
item_similarity = pd.DataFrame(cosine_sim_matrix, index=R_items.index, columns=R_items.index)
valid_sim_items = set(item_similarity.index)

user_history_norm = {
    u: {i: r for i, r in ratings.items() if not pd.isna(r)}
    for u, ratings in R_double_norm.T.to_dict().items()
}

def predict_double_norm_k14(row):
    user = row["User"]
    target_item = row["Restaurant"]

    baseline_pred = global_mean + u_bias_dict[user] + i_bias_dict[target_item]

    ratings_norm = user_history_norm[user]
    valid_items = [i for i in ratings_norm if i in valid_sim_items]

    sims = item_similarity.loc[target_item, valid_items]
    sims = sims[sims > 0]

    if sims.empty:
        return baseline_pred

    top_k = sims.nlargest(14)
    numerator = sum(sim_score * ratings_norm[item] for item, sim_score in zip(top_k.index, top_k))
    denominator = top_k.sum()

    if denominator == 0:
        return baseline_pred

    return baseline_pred + (numerator / denominator)

print("Generando predicciones...")
targets = pd.read_csv("target_recommendations.csv", sep=";", header=None, names=["User", "Restaurant", "Real"])
preds = targets.apply(predict_double_norm_k14, axis=1)
preds = preds.clip(-10, 10)

# Guardar y preparar para compute_mae
output_path = "resultats_knn_double_norm_cosine.csv"
result_df = pd.DataFrame({
    "User": targets["User"],
    "Restaurant": targets["Restaurant"],
    "Pred": preds
})
result_df.to_csv(output_path, sep=";", index=False, header=False, float_format="%.6f")

elapsed = time.time() - start_time
print(f"¡Hecho! Archivo: {output_path}")
print(f"Tiempo total: {elapsed:.1f} segundos.")

# Validación directa usando compute_mae.py
print("\nEjecutando compute_mae.py …")
get_ipython().system(f"python compute_mae.py target_recommendations.csv {output_path}")


=== MÉTODO 8: KNN (k=14) CON DOBLE NORMALIZACIÓN (Similitud del coseno) ===
Calculando sesgos globales...
Aplicando doble normalización a la matriz...
Calculando similitud coseno sobre la matriz normalizada...
Generando predicciones...
¡Hecho! Archivo: resultats_knn_double_norm_cosine.csv
Tiempo total: 65.9 segundos.

Ejecutando compute_mae.py …
MAE: 3.131
